# 02 · EDA: windowed sEMG features, dimensionality and embeddings

Notebook version of `eda/make_figures.py`. It produces the four figures in `eda/README.md`
and the numbers quoted there.

**The question this notebook answers:** is the sEMG feature space low-dimensional, and do
nonlinear embeddings (UMAP, PHATE) separate movement tasks better than a linear one (PCA)?

**Pipeline**
1. Cut each recording into 250 ms windows and turn each window into 80 features (`eda/features.py`)
2. Raw-signal and spectrum sanity checks (figures 1–2)
3. PCA: how many dimensions does the data actually use? (figure 3)
4. PCA vs UMAP vs PHATE in 2D, and how well each separates *stage* and *hand* (figure 4)

Runs on Colab and locally. Figures are shown inline and also saved to `eda/figures/`.

In [ ]:
# Environment setup: works on Colab and locally.
# After this cell the working directory is the repo root, and `eda/` is importable.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        # Our fork: it contains eda/ (Meta's upstream repo does not)
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py umap-learn phate
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "eda"))

# Same dataset location as notebook 03 (repo root). Falls back to ~/ where eda/download_data.sh puts it.
DATA_DIR = ROOT / "emg2pose_dataset_mini"
if not DATA_DIR.exists() and (Path.home() / "emg2pose_dataset_mini").exists():
    DATA_DIR = Path.home() / "emg2pose_dataset_mini"
if not DATA_DIR.exists():
    url = "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar"
    subprocess.run(["curl", "-s", url, "-o", "emg2pose_dataset_mini.tar"], check=True)
    subprocess.run(["tar", "-xf", "emg2pose_dataset_mini.tar"], check=True)
    os.remove("emg2pose_dataset_mini.tar")

print(f"{'Colab' if IN_COLAB else 'Local'} | repo root: {ROOT} | data: {DATA_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from features import BANDS, SAMPLE_RATE, build_matrix, read_session  # eda/features.py

FIG_DIR = ROOT / "eda" / "figures"
CACHE = ROOT / "eda" / ".cache"
FIG_DIR.mkdir(exist_ok=True)
N_EMBED_SAMPLE = 6000
RANDOM_SEED = 0

plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

## 1. From raw sEMG to a feature matrix

Raw sEMG is 2,000 samples/s × 16 channels, which is too fast and noisy to compare directly.
`build_matrix` (in `eda/features.py`) cuts every recording into **250 ms windows with a 125 ms hop**
(50% overlap). For each window and each channel it computes:

- log RMS amplitude (how active the muscle is)
- log power in 4 frequency bands: 20–50, 50–100, 100–200, 200–450 Hz

16 channels × 5 features = **80 features per window**. Each window also keeps its recording's
labels (stage, hand side), which are stored as attributes in each HDF5 file.

> **Read the code:** open `eda/features.py` and find `window_features`. Why is a Hann taper
> applied before the FFT? Why take the log of power? Write your answers here before moving on.

In [ ]:
X, _, metas, sess = build_matrix(DATA_DIR)          # X: (n_windows, 80); sess: recording index per window
Xs = StandardScaler().fit_transform(X)              # z-score each feature so no feature dominates by scale
stage_all = np.array([metas[i].stage for i in sess])
side_all = np.array([metas[i].side for i in sess])

print(f"Feature matrix: {X.shape}  |  recordings: {len(metas)}  |  "
      f"stages: {len(set(stage_all))}  |  users: {len({m.user for m in metas})}")

## 2. Sanity checks: what the raw signal looks like (figure 1)

Four seconds of one recording: raw sEMG (top), a 50 ms RMS envelope (middle) and five joint
angles (bottom). The point is to see that envelope bursts line up with joint movement.

In [ ]:
path = sorted(DATA_DIR.glob("*.hdf5"))[0]
emg, angles, meta = read_session(path)

t0, t1 = int(10 * SAMPLE_RATE), int(14 * SAMPLE_RATE)
t = np.arange(t0, t1) / SAMPLE_RATE

fig, axes = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)

offset = 0.0
step = float(np.abs(emg[t0:t1]).max()) * 2.2
for ch in range(4):
    axes[0].plot(t, emg[t0:t1, ch] + offset, lw=0.4)
    axes[0].text(t[0] - 0.12, offset, f"ch{ch}", va="center", ha="right", fontsize=7)
    offset += step
axes[0].set_ylabel("sEMG (a.u.)")
axes[0].set_title(f"Raw sEMG and paired hand pose — user {meta.user[:6]}…, "
                  f"stage {meta.stage}, {meta.side} hand, {meta.sample_rate:.0f} Hz", fontsize=10)
axes[0].set_yticks([])

win = int(0.05 * SAMPLE_RATE)
kernel = np.ones(win) / win
for ch in range(4):
    env = np.sqrt(np.convolve(emg[t0:t1, ch].astype(np.float64) ** 2, kernel, "same"))
    axes[1].plot(t, env, lw=0.9, label=f"ch{ch}")
axes[1].set_ylabel("RMS envelope")
axes[1].legend(ncol=4, fontsize=7, loc="upper right")

for j in range(0, 20, 4):
    axes[2].plot(t, angles[t0:t1, j], lw=0.9, label=f"joint {j}")
axes[2].set_ylabel("Joint angle (rad)")
axes[2].set_xlabel("Time (s)")
axes[2].legend(ncol=5, fontsize=7, loc="upper right")

fig.tight_layout()
fig.savefig(FIG_DIR / "01_raw_signal.png", bbox_inches="tight")
plt.show()

## 3. Spectra and channel amplitudes (figure 2)

**Left:** power spectrum of every channel; shaded regions are the feature bands. sEMG power
should sit mostly below ~450 Hz. **Right:** per-channel RMS across all 30 recordings, on a log
scale. Channels differ by more than 10×, which is why features are z-scored before PCA.

In [ ]:
paths = sorted(DATA_DIR.glob("*.hdf5"))
emg, _, meta = read_session(paths[0])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

for ch in range(16):
    freqs, psd = signal.welch(emg[:, ch], fs=SAMPLE_RATE, nperseg=2048)
    axes[0].semilogy(freqs, psd, lw=0.7, alpha=0.75)
for lo, hi in BANDS:
    axes[0].axvspan(lo, hi, alpha=0.06, color="k")
axes[0].set_xlim(0, 500)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("PSD (a.u.²/Hz)")
axes[0].set_title("Power spectrum, all 16 channels\n(shaded = feature bands)", fontsize=9)

rms = np.zeros((len(paths), 16))
for i, p in enumerate(paths):
    e, _, _ = read_session(p)
    rms[i] = np.sqrt(np.mean(e.astype(np.float64) ** 2, axis=0))

axes[1].boxplot([rms[:, c] for c in range(16)], widths=0.6, flierprops={"markersize": 2})
axes[1].set_yscale("log")
axes[1].set_xlabel("Channel")
axes[1].set_ylabel("RMS amplitude (a.u.)")
axes[1].set_title(f"Per-channel RMS across {len(paths)} sessions\n(one user, 15 stages, both hands)",
                  fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "02_spectra_channels.png", bbox_inches="tight")
plt.show()

## 4. How many dimensions does the data use? (figure 3)

PCA finds orthogonal directions ordered by variance. Two summaries:

- **Cumulative variance:** how many components you need for 80/90/95% of the variance.
- **Participation ratio** $PR = (\sum_i \lambda_i)^2 / \sum_i \lambda_i^2$, where $\lambda_i$ are the
  PCA eigenvalues. If all variance is in one component, PR = 1; if spread equally over $d$ components, PR = $d$.
  It is a single-number "effective dimensionality".

> **Think:** PCA can only find *flat* subspaces. If the data lay on a curved 2-D sheet, what
> would PCA's participation ratio over-estimate or under-estimate?

In [ ]:
pca = PCA().fit(Xs)
ev = pca.explained_variance_ratio_
cum = np.cumsum(ev)
lam = pca.explained_variance_
pr = float(lam.sum() ** 2 / (lam ** 2).sum())

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))

axes[0].bar(np.arange(1, 21), ev[:20] * 100, color="#3b6ea5")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Variance explained (%)")
axes[0].set_title(f"Scree — PC1 alone is {ev[0]*100:.0f}%", fontsize=9)

axes[1].plot(np.arange(1, len(cum) + 1), cum * 100, "o-", ms=3, color="#3b6ea5")
for thresh, colour in [(0.80, "#c44"), (0.90, "#e88"), (0.95, "#aaa")]:
    n = int(np.searchsorted(cum, thresh) + 1)
    axes[1].axhline(thresh * 100, ls="--", lw=0.8, color=colour)
    axes[1].annotate(f"{int(thresh*100)}% → {n} PCs", (n, thresh * 100),
                     textcoords="offset points", xytext=(8, -11), fontsize=7.5, color=colour)
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance (%)")
axes[1].set_title(f"80 features → participation ratio {pr:.1f}", fontsize=9)
axes[1].set_xlim(0, 40)

fig.tight_layout()
fig.savefig(FIG_DIR / "03_pca_dimensionality.png", bbox_inches="tight")
plt.show()
print(f"Participation ratio: {pr:.2f}")

## 5. Nonlinear embeddings: UMAP and PHATE

Both run on a fixed random subsample of 6,000 windows (seed 0) because they scale poorly.
Results are cached in `eda/.cache/` so re-running is instant. Delete that folder to recompute.

- **UMAP** builds a nearest-neighbour graph and lays it out to preserve local neighbourhoods,
  which tends to form tight clusters.
- **PHATE** diffuses over the neighbour graph, which tends to preserve continuous transitions
  and branching (trajectory-like structure).

In [ ]:
CACHE.mkdir(exist_ok=True)
paths_cache = {k: CACHE / f"{k}.npy" for k in ("idx", "umap", "phate")}

if all(p.exists() for p in paths_cache.values()):
    idx, U, P = (np.load(p) for p in paths_cache.values())
    print("Loaded cached embeddings")
else:
    import umap, phate
    rng = np.random.default_rng(RANDOM_SEED)
    n = min(N_EMBED_SAMPLE, Xs.shape[0])
    idx = np.sort(rng.choice(Xs.shape[0], n, replace=False))
    subset = Xs[idx]
    print(f"Computing UMAP on {n} windows…")
    U = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=RANDOM_SEED).fit_transform(subset)
    print(f"Computing PHATE on {n} windows…")
    P = phate.PHATE(n_jobs=-1, random_state=RANDOM_SEED, verbose=0).fit_transform(subset)
    np.save(paths_cache["idx"], idx); np.save(paths_cache["umap"], U); np.save(paths_cache["phate"], P)

pcs = pca.transform(Xs)[idx][:, :2]   # PCA's first 2 components on the same subsample

## 6. Measuring separation

A picture can look separated without being separated, so each 2-D embedding gets a number:
**15-nearest-neighbour classification accuracy, 5-fold cross-validation**, for stage
(chance ≈ 6.7% with 15 stages) and for hand side (chance 50%).

> **Known flaw (worth fixing yourself):** windows overlap by 50% and neighbouring windows are
> nearly identical, so with random folds a test window's nearest neighbours are often its own
> neighbours in time. That inflates accuracy. A fairer test holds out whole recordings, e.g.
> `GroupKFold` with `groups=sess[idx]`. Note that with one user, each stage has only two
> recordings (left and right hand), so think about what holding one out actually tests.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

accs, side_accs = {}, {}
for name, emb in [("PCA", pcs), ("UMAP", U), ("PHATE", P)]:
    accs[name] = float(cross_val_score(KNeighborsClassifier(15), emb, stage_all[idx], cv=5).mean())
    side_accs[name] = float(cross_val_score(KNeighborsClassifier(15), emb, side_all[idx], cv=5).mean())
    print(f"{name:6s} stage acc {accs[name]*100:5.1f}%   hand acc {side_accs[name]*100:5.1f}%")

## 7. PCA vs UMAP vs PHATE (figure 4)

Top row coloured by stage, bottom row by hand side, with the accuracies from above in each title.

In [ ]:
stage, side = stage_all[idx], side_all[idx]
stages = sorted(set(stage))
cmap = plt.get_cmap("tab20")
colours = {s: cmap(i % 20) for i, s in enumerate(stages)}

fig, axes = plt.subplots(2, 3, figsize=(12.5, 7.4))
for col, (name, emb) in enumerate([("PCA", pcs), ("UMAP", U), ("PHATE", P)]):
    ax = axes[0, col]
    for s in stages:
        m = stage == s
        ax.scatter(emb[m, 0], emb[m, 1], s=1.6, color=colours[s], alpha=0.55, linewidths=0, rasterized=True)
    ax.set_title(f"{name} — coloured by stage\nstage kNN accuracy "
                 f"{accs[name]*100:.0f}% (chance {100/len(stages):.0f}%)", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

    ax = axes[1, col]
    for s, c in [("left", "#2b7bba"), ("right", "#d1622b")]:
        m = side == s
        ax.scatter(emb[m, 0], emb[m, 1], s=1.6, color=c, alpha=0.5, linewidths=0, label=s, rasterized=True)
    ax.set_title(f"{name} — coloured by hand\nhand kNN accuracy {side_accs[name]*100:.0f}% (chance 50%)",
                 fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    if col == 0:
        ax.legend(markerscale=6, fontsize=8, loc="best")

handles = [plt.Line2D([], [], marker="o", ls="", ms=4, color=colours[s], label=s) for s in stages]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=7, frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Windowed sEMG features in 2D — task structure dominates, but hand side leaks in too", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIG_DIR / "04_embeddings.png", bbox_inches="tight")
plt.show()

## Questions to answer before moving on

1. Re-run section 6 with `GroupKFold` over recordings. Which method's accuracy drops the most, and why?
2. The raw 80-D features get 49.6% stage accuracy (see `eda/README.md`). What does it mean that
   UMAP-2D gets 40% but PCA-2D only 24%?
3. Does a 2-D embedding being "separated" tell you anything about the *trajectories* within a stage?
   What would you need to compute to find out?